# Week 2 — Monte Carlo Prices and Their Uncertainty
## Overview

Last week we defined the gross GMAB top-up. This week we specify a risk-neutral model for the maturity account, turn random account draws into a price estimate, and measure the estimate's simulation uncertainty. We also build the direct-all Monte Carlo baseline: simulate every contract in a teaching portfolio.

**By the end, you should be able to:**

- Generate a terminal account from a lognormal model and explain the role of each parameter.
- Derive the Monte Carlo mean and the standard error of that mean.
- Interpret an approximate confidence interval without treating it as a guaranteed error bound.
- Use repeated independent simulations to study how estimation error changes with path count.
- Keep training, validation, and audit randomness separate.
- Report the path work and elapsed time of the direct-all baseline.

**Prerequisites:** Week 1; expectation and variance; Python arrays and table columns. We use a terminal distribution directly. Stochastic calculus and a proof of Itô's formula are not prerequisites.

**Concrete outputs:** a terminal-account histogram; one Monte Carlo price with an approximate interval; an independent-replication error table and plot; and a direct-all portfolio baseline table with cost information.

**Time plan:** 4–6 independent hours plus a 75-minute meeting. The default worked code should take a minute or two on an ordinary laptop. The core portfolio contains 1,200 contracts; do not switch to 10,000-contract research mode until the small version runs correctly.

**Target reminder:** $y=\mathbb E_Q[e^{-rT}(g-A_T/P)^+]$ at a specified fee; dollars are $L=P y$. We assume survival to maturity and apply no survival-probability weighting. This gross guarantee liability excludes mortality, lapses, withdrawals, ratchets, and subtraction of fee income.

### Setup and complete simulator reference

Run the next two cells once. The visible reference cell contains the complete functions used in this notebook, so no other project notebook or local source file is needed. The external Python packages are NumPy, pandas, SciPy, matplotlib, and scikit-learn; they must already be installed in your notebook environment. No data download occurs.

For today's guided work, focus on `make_portfolio`, `gmab_mc`, and `gmab_exact`. `gmab_mc` returns a table with the normalized price estimate, its standard error, the payoff standard deviation, and the number of paths. `gmab_exact` is an analytical **audit** helper, not a source of training labels.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from time import perf_counter
try:
    from IPython.display import display
except ImportError:
    def display(*objects):
        for obj in objects:
            print(obj)

pd.set_option("display.precision", 6)
r = 0.03

In [ ]:
# Complete common pricing code. No data files or earlier notebooks are required.
import sys
import json
import platform
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

try:
    from IPython.display import display
except ImportError:
    def display(*objects):
        for obj in objects:
            print(obj)

RISK_FREE_RATE = 0.03
FEATURES = ['account_ratio', 'guarantee_ratio', 'maturity', 'volatility', 'fee']
plt.rcParams.update({'figure.figsize': (7.2, 4.3), 'axes.grid': True,
                     'grid.alpha': 0.22, 'font.size': 10})


def make_portfolio(n=1200, seed=2027):
    """Synthetic in-force GMAB contracts, never customer data.

    P is a reference premium, A0=P*account_ratio, and G=P*guarantee_ratio.
    T is the remaining term. All contracts share the same risk-free rate.
    Volatility describes the contract's fund; rates are annual decimals.
    """
    if not isinstance(n, (int, np.integer)) or n < 10:
        raise ValueError('n must be an integer of at least 10.')
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        'contract_id': np.arange(n, dtype=int),
        'premium': rng.uniform(10_000.0, 50_000.0, n),
        'account_ratio': rng.uniform(0.65, 1.40, n),
        'guarantee_ratio': rng.uniform(0.90, 1.60, n),
        'maturity': rng.integers(5, 21, n).astype(float),
        'volatility': rng.uniform(0.12, 0.35, n),
        'fee': rng.uniform(0.005, 0.030, n),
    })


def validate_contracts(df):
    required = ['contract_id', 'premium'] + FEATURES
    missing = [c for c in required if c not in df]
    if missing:
        raise ValueError(f'Missing contract columns: {missing}')
    if len(df) == 0:
        raise ValueError('Supply at least one contract.')
    values = df[required].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError('All contract fields must be finite.')
    ids = df['contract_id'].to_numpy(dtype=float)
    if (ids < 0).any() or (ids != np.floor(ids)).any() or len(np.unique(ids)) != len(ids):
        raise ValueError('contract_id must contain unique nonnegative integers.')
    if (df[['premium', 'account_ratio', 'guarantee_ratio']] <= 0).any().any():
        raise ValueError('Premium and account/guarantee ratios must be positive.')
    if (df[['maturity', 'volatility', 'fee']] < 0).any().any():
        raise ValueError('Maturity, volatility and fee must be nonnegative.')


def gmab_mc(df, n_paths=2048, seed=123):
    """Independent plain Monte Carlo for per-premium gross top-up liability.

    Identical (seed, contract_id) reproduces the same path stream even when
    the contract appears in a different subset or row order. Distinct IDs
    give independent streams. This is NOT a common-market portfolio-risk
    simulator: it estimates a sum of expected liabilities, not portfolio VaR.
    SE describes Monte Carlo sampling uncertainty only.
    """
    validate_contracts(df)
    if not isinstance(n_paths, (int, np.integer)) or n_paths < 2:
        raise ValueError('n_paths must be an integer of at least 2.')
    if not isinstance(seed, (int, np.integer)) or seed < 0:
        raise ValueError('seed must be a nonnegative integer.')
    values, errors, sds = [], [], []
    for row in df.itertuples():
        rng = np.random.default_rng(np.random.SeedSequence([int(seed), int(row.contract_id)]))
        z = rng.standard_normal(n_paths)
        terminal_ratio = row.account_ratio * np.exp(
            (RISK_FREE_RATE - row.fee - 0.5 * row.volatility**2) * row.maturity
            + row.volatility * np.sqrt(row.maturity) * z
        )
        payoff = np.exp(-RISK_FREE_RATE * row.maturity) * np.maximum(
            row.guarantee_ratio - terminal_ratio, 0.0)
        sd = float(np.std(payoff, ddof=1))
        values.append(float(np.mean(payoff)))
        errors.append(sd / np.sqrt(n_paths))
        sds.append(sd)
    return pd.DataFrame({'value': values, 'se': errors, 'payoff_sd': sds,
                         'paths': int(n_paths)}, index=df.index)


def gmab_exact(df):
    """Closed-form put benchmark under the simplified model: audit only.

    Never use these values as training or tuning labels in the project.
    The deterministic cases T=0 or sigma=0 are handled separately.
    """
    validate_contracts(df)
    a = df['account_ratio'].to_numpy(dtype=float)
    g = df['guarantee_ratio'].to_numpy(dtype=float)
    t = df['maturity'].to_numpy(dtype=float)
    sig = df['volatility'].to_numpy(dtype=float)
    fee = df['fee'].to_numpy(dtype=float)
    ans = np.maximum(g * np.exp(-RISK_FREE_RATE*t) - a * np.exp(-fee*t), 0.0)
    regular = (t > 0) & (sig > 0)
    if np.any(regular):
        tr, sr = t[regular], sig[regular]
        d1 = (np.log(a[regular] / g[regular]) +
              (RISK_FREE_RATE-fee[regular]+0.5*sr**2)*tr) / (sr*np.sqrt(tr))
        d2 = d1 - sr*np.sqrt(tr)
        ans[regular] = (g[regular]*np.exp(-RISK_FREE_RATE*tr)*norm.cdf(-d2)
                        - a[regular]*np.exp(-fee[regular]*tr)*norm.cdf(-d1))
    return np.maximum(ans, 0.0)


def split_portfolio(df, seed=2027):
    """Split contracts before building pairs, selecting labels or tuning."""
    validate_contracts(df)
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(df))
    n_candidate, n_validation = int(0.60*len(df)), int(0.20*len(df))
    if min(n_candidate, n_validation, len(df)-n_candidate-n_validation) < 1:
        raise ValueError('Portfolio is too small for the requested split.')
    return {
        'candidate': df.iloc[order[:n_candidate]].copy(),
        'validation': df.iloc[order[n_candidate:n_candidate+n_validation]].copy(),
        'test': df.iloc[order[n_candidate+n_validation:]].copy(),
    }


def score_predictions(y_true, y_pred, premium=None):
    """Errors per premium, plus signed aggregate relative error when requested."""
    truth = np.asarray(y_true, dtype=float).reshape(-1)
    pred = np.asarray(y_pred, dtype=float).reshape(-1)
    if truth.shape != pred.shape or len(truth) == 0:
        raise ValueError('Truth and predictions must have equal nonzero length.')
    if not (np.isfinite(truth).all() and np.isfinite(pred).all()):
        raise ValueError('Truth and predictions must be finite.')
    error = pred-truth
    out = {'rmse': float(np.sqrt(np.mean(error**2))),
           'mae': float(np.mean(np.abs(error))),
           'p95_abs': float(np.quantile(np.abs(error), 0.95)),
           'bias': float(np.mean(error)),
           'negative_fraction': float(np.mean(pred < 0))}
    if premium is not None:
        p = np.asarray(premium, dtype=float).reshape(-1)
        if p.shape != truth.shape or not np.isfinite(p).all() or np.any(p <= 0):
            raise ValueError('Premium weights must be finite, positive and aligned.')
        denominator = float(p @ truth)
        out['portfolio_relative_error'] = (float(p @ error)/denominator
                                            if abs(denominator) > 1e-12 else np.nan)
    return out


## 1. A risk-neutral terminal-account model

Under the model's risk-neutral law $Q$, the normalized terminal account is

$$X_T=\frac{A_T}{P}
=a\exp\!\left[\left(r-\phi-\frac{\sigma^2}{2}\right)T+\sigma\sqrt{T}\,Z\right],
\qquad Z\sim N(0,1).$$

This is the terminal distribution of a geometric Brownian motion account with a continuous proportional fee. For this one-maturity payoff, we need only one normal draw per path: we do not need a daily time grid.

The logarithm of $X_T$ is normal, so $X_T$ is positive and lognormal. The term $-\sigma^2 T/2$ makes the mean satisfy

$$\mathbb E_Q[X_T]=a e^{(r-\phi)T}.$$

Indeed, $\mathbb E[e^{uZ}]=e^{u^2/2}$ for a standard normal variable, so the positive $\sigma^2T/2$ from averaging the exponential cancels the negative term in the exponent. It follows that the discounted expected account is $a e^{-\phi T}$.

**Interpretation:** $r-\phi$ is the account drift under this valuation model. It is not our forecast of the real-world equity return. Historical scenario probabilities and risk-neutral valuation probabilities answer different questions. We also treat $r$, $\sigma$, and $\phi$ as constant for each contract. With all else and the normal draw held fixed, increasing $\phi$ lowers $X_T$ and therefore weakly **increases the gross top-up**. This differs from valuing the entire account-plus-guarantee payment or solving a fair-fee equation.

The example below draws terminal accounts for one fixed contract. The theoretical mean is a diagnostic reference; a finite-sample mean will fluctuate around it.

In [ ]:
example_contract = pd.DataFrame({
    "contract_id": [20001],
    "premium": [25000.0],
    "account_ratio": [1.0],
    "guarantee_ratio": [1.0],
    "maturity": [10.0],
    "volatility": [0.20],
    "fee": [0.01],
})
validate_contracts(example_contract)
example = example_contract.iloc[0]
a, g, T, sigma, phi = [float(example[name]) for name in
                       ["account_ratio", "guarantee_ratio", "maturity", "volatility", "fee"]]

terminal_rng = np.random.default_rng(21001)
terminal_z = terminal_rng.standard_normal(20000)
terminal_account_ratio = a * np.exp((r - phi - 0.5 * sigma**2) * T
                                     + sigma * np.sqrt(T) * terminal_z)
theoretical_account_mean = a * np.exp((r - phi) * T)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(terminal_account_ratio, bins=70, density=True, alpha=0.75, label="Risk-neutral draws")
ax.axvline(theoretical_account_mean, color="black", linestyle="--", label="Model mean")
ax.axvline(g, color="tab:red", linestyle=":", label="Guarantee ratio")
ax.set(xlabel="Terminal account / premium", ylabel="Density",
       title="One contract's risk-neutral terminal-account distribution")
ax.legend()
fig.tight_layout()
plt.show()

display(pd.DataFrame({
    "quantity": ["Sample account-ratio mean", "Theoretical account-ratio mean",
                 "Sample fraction below guarantee"],
    "value": [terminal_account_ratio.mean(), theoretical_account_mean,
              np.mean(terminal_account_ratio < g)],
}))

**Interpretation checkpoint.** A path below the guarantee produces a positive top-up. Is the probability of being below the guarantee itself a price? Explain what information about the size and timing of the payment that probability leaves out.

## 2. From independent payoffs to a mean, standard error, and interval

For $N$ independent standard normal draws, compute

$$Y_b=e^{-rT}\max(g-X_T^{(b)},0),\qquad b=1,\ldots,N.$$

The Monte Carlo estimate is

$$\widehat y_N=\frac1N\sum_{b=1}^N Y_b.$$

For independent draws, $\mathbb E[\widehat y_N]=y$ and

$$\operatorname{Var}(\widehat y_N)=\frac{\operatorname{Var}(Y)}{N}.$$

Estimate the payoff variance using the sample variance with denominator $N-1$:

$$s_Y^2=\frac{1}{N-1}\sum_{b=1}^N(Y_b-\widehat y_N)^2,
\qquad \widehat{\operatorname{SE}}(\widehat y_N)=\frac{s_Y}{\sqrt N}.$$

The payoff standard deviation describes variation **between simulated outcomes**. The standard error describes uncertainty in **their estimated mean**. They are not interchangeable. In NumPy, `std(ddof=1)` supplies the $N-1$ convention.

A usual large-sample, pointwise 95% interval is

$$\widehat y_N\pm1.96\,\widehat{\operatorname{SE}}(\widehat y_N).$$

Under repeated independent use for this fixed contract and fixed path count, such intervals aim for about 95% coverage. They do not guarantee a 95% probability statement about a fixed unknown parameter after the data are observed. Normal approximation can be poor for small samples or rare, highly skewed positive payoffs. Zero observed positive payoffs can even produce an estimated standard error of zero without proving the true liability is zero.

Our payoffs satisfy $0\leq Y\leq e^{-rT}g$, so the moments needed for an ordinary independent-draw central limit theorem exist. Finite variance alone does not ensure a small finite-sample approximation error. Repeatedly inspecting intervals and stopping when one looks favorable also changes the coverage question.

In [ ]:
# Derive the estimator directly from the terminal draws above.
manual_n = 4096
manual_payoffs = np.exp(-r * T) * np.maximum(g - terminal_account_ratio[:manual_n], 0.0)
manual_value = manual_payoffs.mean()
manual_payoff_sd = manual_payoffs.std(ddof=1)
manual_se = manual_payoff_sd / np.sqrt(manual_n)

manual_summary = pd.DataFrame({
    "quantity": ["Price per premium", "Payoff standard deviation", "Standard error of mean",
                 "Approximate 95% interval: lower", "Approximate 95% interval: upper"],
    "value": [manual_value, manual_payoff_sd, manual_se,
              manual_value - 1.96 * manual_se, manual_value + 1.96 * manual_se],
})
display(manual_summary)

# The shared function uses a separate, reproducible stream for this same contract.
mc_example = gmab_mc(example_contract, n_paths=4096, seed=22001)
mc_report = mc_example.copy()
mc_report["ci95_lower"] = mc_report["value"] - 1.96 * mc_report["se"]
mc_report["ci95_upper"] = mc_report["value"] + 1.96 * mc_report["se"]
mc_report["liability_dollars"] = example_contract["premium"] * mc_report["value"]
mc_report["se_dollars"] = example_contract["premium"] * mc_report["se"]
display(mc_report)
print("The manual and shared estimates use different draws and need not be identical.")

### What this interval does and does not quantify

The interval describes Monte Carlo sampling uncertainty under the specified contract and model. It does not account for misspecified volatility, omitted policyholder behavior, uncertain interest-rate assumptions, or future prediction error from a machine learning model.

We show the usual normal interval without silently clipping its lower endpoint to zero. A negative lower endpoint is a limitation of that approximation, even though the true guarantee liability is nonnegative. More careful rare-event interval methods are outside the core project.

**Quick calculation:** because standard error scales as $N^{-1/2}$, reducing it by a factor of two generally requires about four times as many independent paths. Reducing it by a factor of ten generally requires about one hundred times as many.

## 3. Use an analytical audit to study Monte Carlo error

This simple GMAB is mathematically a put-like top-up. When $a,g,\sigma,T>0$, its normalized price is

$$y_{\mathrm{audit}}=g e^{-rT}\Phi(-d_2)-a e^{-\phi T}\Phi(-d_1),$$

$$d_1=\frac{\log(a/g)+(r-\phi+\sigma^2/2)T}{\sigma\sqrt T},
\qquad d_2=d_1-\sigma\sqrt T.$$

Here $\Phi$ is the standard normal distribution function. You do not need to derive this option-pricing formula for the core project. It lets us check whether the simulator implements our stated target correctly.

**Audit boundary:** the analytical value is used below only to measure simulation error. In later prediction experiments, both reference-contract labels and validation labels come from Monte Carlo. Analytical test values are revealed only after model choices have been frozen. Otherwise our supposedly limited simulation budget would not describe the experiment we actually ran.

One simulation error can be unusually large or small. At each path count $N$, we therefore repeat the experiment $R$ times with independent seeds and calculate

$$\operatorname{RMSE}_N=\sqrt{\frac1R\sum_{j=1}^R
(\widehat y_{N,j}-y_{\mathrm{audit}})^2}.$$

The predicted order is $N^{-1/2}$. The plotted reference line has that slope; it is not a fitted guarantee that every observed point will follow it. With only a few dozen replications, empirical coverage rates and RMSEs themselves have sampling uncertainty.

In [ ]:
audit_truth = float(gmab_exact(example_contract)[0])
audit_path_counts = np.array([256, 1024, 4096, 16384])
audit_repetitions = 48

audit_records = []
audit_start = perf_counter()
for budget_index, n_paths in enumerate(audit_path_counts):
    for replicate in range(audit_repetitions):
        # Changing this seed makes a new independent simulation replication.
        estimate = gmab_mc(example_contract, n_paths=int(n_paths),
                           seed=230000 + 1000 * budget_index + replicate).iloc[0]
        error = float(estimate["value"]) - audit_truth
        se = float(estimate["se"])
        audit_records.append({
            "paths": int(n_paths), "replicate": replicate,
            "estimate": float(estimate["value"]), "error": error,
            "reported_se": se,
            "covered": abs(error) <= 1.96 * se,
        })
audit_elapsed = perf_counter() - audit_start
audit_replicates = pd.DataFrame(audit_records)

audit_rows = []
for n_paths, group in audit_replicates.groupby("paths", sort=True):
    audit_rows.append({
        "paths": int(n_paths),
        "replications": len(group),
        "rmse_per_premium": np.sqrt(np.mean(group["error"]**2)),
        "mean_reported_se": group["reported_se"].mean(),
        "mean_error": group["error"].mean(),
        "empirical_interval_coverage": group["covered"].mean(),
    })
audit_table = pd.DataFrame(audit_rows)
display(audit_table)
print(f"Analytical audit value: {audit_truth:.6f} per premium")
print(f"Audit experiment elapsed time: {audit_elapsed:.2f} seconds")

In [ ]:
reference_rmse = (audit_table["rmse_per_premium"].iloc[0]
                  * np.sqrt(audit_table["paths"].iloc[0] / audit_table["paths"].to_numpy()))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.loglog(audit_table["paths"], 100 * audit_table["rmse_per_premium"], "o-",
          label="Independent-replication RMSE")
ax.loglog(audit_table["paths"], 100 * audit_table["mean_reported_se"], "s--",
          label="Mean reported standard error")
ax.loglog(audit_table["paths"], 100 * reference_rmse, ":", color="gray",
          label=r"Reference slope $N^{-1/2}$")
ax.set(xlabel="Independent paths per price estimate",
       ylabel="RMSE or SE (percentage points of premium)",
       title="How simulation uncertainty changes with path count")
ax.grid(alpha=0.25, which="both")
ax.legend()
fig.tight_layout()
plt.show()

**Interpretation checkpoint.** Compare the replication RMSE with the mean reported standard error. Explain why they should be of similar size for an unbiased estimator, but need not agree exactly in a finite experiment. If one empirical coverage rate is 0.92, does that alone prove the interval formula is wrong? What additional experiment would be informative?

## 4. Seeds, contract IDs, and independent label streams

A pseudorandom seed makes an experiment reproducible. Reusing a seed with the same contract and path count reproduces the same numerical sample; it does **not** create a second independent replication.

In our shared simulator, each contract receives a stream built from `SeedSequence([seed, contract_id])`. This makes a contract's simulation depend on its identifier and experiment seed, rather than on its current row position. Shuffling the table therefore does not change its labels. Contract IDs must be unique.

Later we will deliberately use different experiment seeds for:

- The Monte Carlo labels of the representative contracts.
- Independently simulated validation labels used to compare candidate models.
- Independently simulated final test labels.
- Separate simulation replications used to study variability.

Different model predictors in one comparison should share the **same representative labels**. That holds the training information fixed. Validation paths should be independent of those labels, so a model is not rewarded for reproducing training simulation noise.

Our implementation draws independent simulation errors across different contracts. That is sufficient for studying expected liability prediction. It is not a joint market-scenario model for portfolio loss distributions, capital, or tail risk.

In [ ]:
seed_demo = make_portfolio(n=10, seed=24001).head(6).copy()
original_labels = gmab_mc(seed_demo, n_paths=512, seed=24002)
shuffled_demo = seed_demo.sample(frac=1.0, random_state=24003)
shuffled_labels = gmab_mc(shuffled_demo, n_paths=512, seed=24002)

original_check = pd.Series(original_labels["value"].to_numpy(),
                           index=seed_demo["contract_id"].to_numpy())
shuffled_check = pd.Series(shuffled_labels["value"].to_numpy(),
                           index=shuffled_demo["contract_id"].to_numpy())
assert np.allclose(original_check.sort_index(), shuffled_check.sort_index())

new_labels = gmab_mc(seed_demo, n_paths=512, seed=24004)
seed_report = pd.DataFrame({
    "contract_id": seed_demo["contract_id"],
    "original_stream": original_labels["value"],
    "new_independent_stream": new_labels["value"],
})
display(seed_report)
print("Reordering preserves each contract's original labels; a new seed changes the sample.")

## 5. The direct-all portfolio baseline and its cost

Let a portfolio contain $n$ contracts. A direct-all Monte Carlo baseline estimates every $y_i$ with $N$ paths and converts the results to dollars:

$$\widehat L_i=P_i\widehat y_i,
\qquad \widehat L_{\mathrm{portfolio}}=\sum_{i=1}^n P_i\widehat y_i.$$

The core simulation work is $nN$ contract-path payoffs. We will count this work as well as wall-clock time because timings depend on hardware, library versions, and background activity.

Because the Monte Carlo errors are independent across contracts in our implementation, an estimated standard error for the **estimated total expected liability** is

$$\widehat{\operatorname{SE}}(\widehat L_{\mathrm{portfolio}})
=\sqrt{\sum_i P_i^2\widehat{\operatorname{SE}}(\widehat y_i)^2}.$$

This is numerical uncertainty in a sum of expected values. It is not the standard deviation of the portfolio's future insurance losses.

The portfolio below uses a separate teaching seed from the future benchmark. In Week 3, we will construct a new portfolio and split contract IDs **before** any model labels are generated. Today's direct-all exercise therefore does not reveal the future benchmark's held-out labels.

There is an analytical solution for this stripped-down model. We intentionally compare simulation methods to learn a workflow that also makes sense when analytical prices are unavailable; these timings do not establish an industrial deployment speedup.

In [ ]:
direct_portfolio = make_portfolio(n=1200, seed=25001)
direct_paths = 2048

direct_start = perf_counter()
direct_labels = gmab_mc(direct_portfolio, n_paths=direct_paths, seed=25002)
direct_elapsed = perf_counter() - direct_start

direct_report = direct_portfolio[["contract_id", "premium"]].copy()
direct_report["value_per_premium"] = direct_labels["value"]
direct_report["se_per_premium"] = direct_labels["se"]
direct_report["liability_dollars"] = direct_report["premium"] * direct_report["value_per_premium"]
direct_report["se_dollars"] = direct_report["premium"] * direct_report["se_per_premium"]

direct_total_liability = float(direct_report["liability_dollars"].sum())
direct_total_se = float(np.sqrt(np.sum(direct_report["se_dollars"]**2)))
direct_cost_table = pd.DataFrame({
    "contracts": [len(direct_portfolio)],
    "paths_per_contract": [direct_paths],
    "contract_path_payoffs": [int(direct_labels["paths"].sum())],
    "simulation_seconds": [direct_elapsed],
    "estimated_total_liability_dollars": [direct_total_liability],
    "estimated_SE_of_total_dollars": [direct_total_se],
})
display(direct_report.head(8))
display(direct_cost_table)
print("This baseline estimates the gross GMAB top-up at the specified contract fees.")

**Research checkpoint.** Suppose a future predictor uses $m$ representative contracts, with $N_{\mathrm{ref}}$ paths each. Its training simulation work is $mN_{\mathrm{ref}}$, but that is not its whole workflow cost: representative selection, fitting, prediction, and any validation simulations also consume resources. What accuracy might be lost when $m$ is small? Which part of the cost accounting should be reported separately from production prediction?

## Reading and discussion

| Source | This week's task | Question to answer |
|---|---|---|
| Mary Hardy (2003), [*Investment Guarantees: Modeling and Risk Management for Equity-Linked Life Insurance*](https://www.wiley-vch.de/en/areas-interest/finance-economics-law/finance-investments-13fi/investments-securities-13fi3/investment-guarantees-978-0-471-39290-3). Wiley. Library or mentor-provided copy. | Read selected passages explaining the investment model and valuation of guarantees; focus on the distinction between modelling outcomes and valuing cash flows. About 20 minutes. | Why must a simulation specify its probability law before an average can be interpreted as a price? |
| Daniel Bauer, Alexander Kling, and Jochen Russ (2008), [*A Universal Pricing Framework for Guaranteed Minimum Benefits in Variable Annuities*](https://doi.org/10.2143/AST.38.2.2033356). | Revisit the model description and the role of discounted guarantee cash flows. Identify one source of complexity that our terminal-only GMAB avoids. About 15 minutes. | Why might a multi-event guarantee need more information than one terminal account value? |
| Hao Quan and Ben (Mingbin) Feng (2026), *Warm-Started Projected Robbins–Monro for Portfolio-Scale Variable Annuity Fee Setting*. Working manuscript supplied by the mentor. | Read the introduction and the definition of the valuation/fair-fee problem. Do not work through the stochastic approximation proof. About 15 minutes. | Which quantity is unknown in fair-fee setting? Which quantity is unknown in our fixed-fee prediction task? |

Submit short answers to the three questions, then explain in your own words why a Monte Carlo confidence interval does not cover model misspecification. The working manuscript is not presented as a published journal article, and no DOI is assumed.

## Coding homework 1 — When can a standard error be misleading?

Use the higher-account contract in the starter. Estimate its price at path counts 256, 1,024, and 4,096. At each count, use at least 30 independent seeds; report the fraction of estimates with zero estimated standard error, the mean reported standard error, and the empirical coverage of the usual 95% interval.

For this **simulator audit exercise only**, use `gmab_exact` to measure coverage. Keep the analytical value in an `audit_` variable and do not turn it into a machine learning label.

Explain what zero observed top-ups tells you, what it does not tell you, and whether the observed coverage patterns support stronger claims than your number of replications allows. A zero-error replication should remain in your table rather than being discarded.

In [ ]:
# Homework starter: extend this single estimate into independent replications.
hw_rare_contract = example_contract.copy()
hw_rare_contract["contract_id"] = 29001
hw_rare_contract["account_ratio"] = 1.35
hw_rare_contract["guarantee_ratio"] = 1.0
hw_rare_contract["maturity"] = 5.0
hw_rare_contract["volatility"] = 0.12
hw_rare_contract["fee"] = 0.005

hw_rare_estimate = gmab_mc(hw_rare_contract, n_paths=256, seed=29002)
audit_rare_truth = float(gmab_exact(hw_rare_contract)[0])
display(hw_rare_estimate)
print(f"Analytical audit reference: {audit_rare_truth:.8f} per premium")
# Add the budget loop, independent replication seeds, and summary table here.

## Coding homework 2 — Report a cost–uncertainty tradeoff

On the same 100-contract portfolio, compare direct-all path budgets of 512, 2,048, and 8,192. Use separate simulation seeds for the budgets. Report the number of contract-path payoffs, elapsed time, root-mean-square per-contract standard error, and the estimated total liability with its numerical standard error.

Draw a cost-versus-standard-error figure. Check whether the change in uncertainty is broadly consistent with the $N^{-1/2}$ scaling, and discuss why wall-clock time need not scale perfectly. Keep the default exercise at 100 contracts; increasing to 10,000 contracts is unnecessary for this learning objective.

In [ ]:
# Homework starter: keep these contracts fixed across your path-budget comparisons.
hw_cost_portfolio = make_portfolio(n=100, seed=29101)
hw_path_counts = [512]
hw_cost_rows = []
for hw_index, hw_paths in enumerate(hw_path_counts):
    hw_start = perf_counter()
    hw_labels = gmab_mc(hw_cost_portfolio, n_paths=hw_paths, seed=29200 + hw_index)
    hw_elapsed = perf_counter() - hw_start
    hw_cost_rows.append({
        "paths": hw_paths,
        "contract_path_payoffs": int(hw_labels["paths"].sum()),
        "seconds": hw_elapsed,
        "rms_SE_per_premium": float(np.sqrt(np.mean(hw_labels["se"]**2))),
    })
display(pd.DataFrame(hw_cost_rows))
# Add the remaining path budgets, total-liability quantities, and the figure here.

## Weekly deliverable and checklist

Submit the completed notebook, both homework tables and figures, and the reading responses. Include a short paragraph recommending a **teaching-scale** path budget based on the uncertainty you observed. Treat it as a provisional choice to revisit, not a universal optimal budget.

- [ ] I can generate $A_T/P$ and explain why the exponent contains $-\sigma^2T/2$.
- [ ] I used plain independent draws and `ddof=1` in the Monte Carlo standard error.
- [ ] I can distinguish payoff variability, estimation uncertainty, and model error.
- [ ] My replication seeds change; rerunning the same seed is not a new replication.
- [ ] I kept analytical prices inside explicitly labelled audit calculations.
- [ ] I reported contract-path work as well as elapsed time.
- [ ] I can explain why the standard error of a portfolio mean is not portfolio loss risk.
- [ ] I restarted the kernel and ran every cell from top to bottom.

**Three-student meeting roles:** simulator explainer, uncertainty checker, and experiment/cost recorder. Rotate from Week 1.